In [1]:
# Cell 1: Dependencies & Hardware Check
!pip install -q transformer-lens torch numpy scipy pandas scikit-learn tabulate

import torch
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Hardware Status: {device.upper()}")
if device == "cuda":
    print(f"🎮 GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"🧠 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.3 MB/s eta 0:00:00
✅ Hardware Status: CUDA
🎮 GPU Model: Tesla T4
🧠 VRAM Available: 15.64 GB


In [2]:
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ==============================================================================
# 0. SERIALIZATION & HELPERS
# ==============================================================================

def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, torch.Tensor): return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None: return obj
    if hasattr(obj, "__dataclass_fields__"): return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_"]:
        if ctx_str.startswith(prefix): return ctx_str[len(prefix):]
    return ctx_str


# ==============================================================================
# 1. ACTION-SENSITIVE UTILITY MATRIX
# ==============================================================================

class DecoupledUtility:
    THRESH_GAIN = 0.005
    THRESH_HAZARD = -0.05
    ALL_ACTIONS = ["APPROVED", "VETO_ZERO_GAIN", "VETO_HAZARD", "VETO_EPISTEMIC"]

    @staticmethod
    def get_utility(action: str, delta_m: float) -> float:
        if delta_m >= DecoupledUtility.THRESH_GAIN:
            if action == "APPROVED": return float(delta_m)
            if action == "VETO_EPISTEMIC": return -0.005
            if action == "VETO_ZERO_GAIN": return -0.015
            if action == "VETO_HAZARD": return -0.03
        elif delta_m < DecoupledUtility.THRESH_HAZARD:
            if action == "VETO_HAZARD": return 0.05
            if action == "VETO_EPISTEMIC": return 0.03
            if action == "VETO_ZERO_GAIN": return 0.01
            if action == "APPROVED": return float(10.0 * delta_m)
        else:
            if action == "VETO_ZERO_GAIN": return 0.005
            if action == "VETO_EPISTEMIC": return 0.002
            if action == "VETO_HAZARD": return 0.0
            if action == "APPROVED": return float(delta_m)
        return -1.0

    @staticmethod
    def get_omniscient_oracle(delta_m: float) -> Tuple[str, float]:
        utils = [DecoupledUtility.get_utility(a, delta_m) for a in DecoupledUtility.ALL_ACTIONS]
        best_idx = int(np.argmax(utils))
        return DecoupledUtility.ALL_ACTIONS[best_idx], utils[best_idx]


# ==============================================================================
# 2. BAYESIAN SCM: STRICT PARAMETRIC EPISTEMIC + RESIDUAL ALEATORIC
# ==============================================================================

@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class PureEpistemicSelfGraph:
    def __init__(self, prior_variance: float = 0.5, base_aleatoric: float = 0.005):
        self.prior_variance = prior_variance
        self.base_aleatoric = base_aleatoric
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}

    def init_node(self, target_key: str, ctx_str: str):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            self.component_models[target_key][family] = {
                "mu": np.zeros(2),
                "precision": np.eye(2) / self.prior_variance,
                "cov": np.eye(2) * self.prior_variance,
                "aleatoric_var": self.base_aleatoric,
                "history": [],
                "n": 0
            }

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float):
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        node["history"].append((dose, delta_obs))

        if len(node["history"]) >= 3:
            y_vals = np.array([h[1] for h in node["history"]])
            diffs = np.diff(y_vals)
            node["aleatoric_var"] = max(self.base_aleatoric, float(0.5 * np.mean(diffs**2)))

        inv_noise = 1.0 / max(node["aleatoric_var"], 1e-6)
        prec_old = node["precision"]
        mu_old = node["mu"]

        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float, float, float]:
        """
        محاسبه خالص آماری بدون دست‌کاری هیوریستیک برون‌یابی:
        (pred_mu, sig_pred, sig_epi, sig_ale)
        """
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        var_epi = 0.0
        var_ale = 0.0

        for t in targets:
            if t.dose == 0.0: continue
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            x_target = np.array([t.dose, t.dose**2])
            pred_delta += float(x_target @ node["mu"])
            var_epi += float(x_target @ node["cov"] @ x_target)
            var_ale += float(node["aleatoric_var"])

        total_var = var_epi + var_ale
        return pred_delta, float(np.sqrt(total_var)), float(np.sqrt(var_epi)), float(np.sqrt(var_ale))


# ==============================================================================
# 3. EMPIRICAL CALIBRATOR
# ==============================================================================

class ScaledEmpiricalCalibrator:
    def __init__(self):
        self.model = LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=300, C=1.0)
        self.is_fitted = False
        self.features: List[np.ndarray] = []
        self.labels: List[int] = []

    @staticmethod
    def extract_vector(mu: float, sig_pred: float, sig_epi: float) -> np.ndarray:
        sig_pred = max(sig_pred, 1e-6)
        z_gain = (mu - DecoupledUtility.THRESH_GAIN) / sig_pred
        z_haz = (DecoupledUtility.THRESH_HAZARD - mu) / sig_pred
        return np.array([mu, sig_pred, sig_epi, z_gain, z_haz])

    def store_sample(self, mu: float, sig_pred: float, sig_epi: float, true_delta: float):
        vec = self.extract_vector(mu, sig_pred, sig_epi)
        if true_delta >= DecoupledUtility.THRESH_GAIN: y = 0
        elif true_delta < DecoupledUtility.THRESH_HAZARD: y = 2
        else: y = 1
        self.features.append(vec)
        self.labels.append(y)

    def train_calibrator(self) -> bool:
        if len(self.labels) < 12 or len(set(self.labels)) < 3:
            self.is_fitted = False
            return False
        X = np.array(self.features)
        y = np.array(self.labels)
        self.model.fit(X, y)
        self.is_fitted = True
        return True

    def predict_beliefs(self, mu: float, sig_pred: float, sig_epi: float) -> Dict[str, float]:
        sig_pred = max(sig_pred, 1e-6)
        if not self.is_fitted:
            p_haz = float(norm.cdf((DecoupledUtility.THRESH_HAZARD - mu) / sig_pred))
            p_gain = float(1.0 - norm.cdf((DecoupledUtility.THRESH_GAIN - mu) / sig_pred))
            p_zero = max(0.0, 1.0 - p_haz - p_gain)
            return {"P_gain": p_gain, "P_zero": p_zero, "P_hazard": p_haz}

        vec = self.extract_vector(mu, sig_pred, sig_epi).reshape(1, -1)
        probs = self.model.predict_proba(vec)[0]
        classes = self.model.classes_

        p_gain = float(probs[np.where(classes == 0)[0][0]]) if 0 in classes else 0.0
        p_zero = float(probs[np.where(classes == 1)[0][0]]) if 1 in classes else 0.0
        p_hazard = float(probs[np.where(classes == 2)[0][0]]) if 2 in classes else 0.0

        total = p_gain + p_zero + p_hazard
        if total <= 0: total = 1.0
        return {"P_gain": p_gain / total, "P_zero": p_zero / total, "P_hazard": p_hazard / total}


# ==============================================================================
# 4. MECHANISTIC SENSORS INTERFACE
# ==============================================================================

class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(
        self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None, noise_sigma: float = 0.0
    ) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets: return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        raw_delta = float(torch.mean(int_m - clean_m).item())

        if noise_sigma > 0.0:
            return raw_delta + float(np.random.normal(0.0, noise_sigma))
        return raw_delta


# ==============================================================================
# 5. CAUSALLY ISOLATED PROBING CONTROLLER (M17.17)
# ==============================================================================

class IsolatedPolicyEngine:
    def __init__(self, scm: PureEpistemicSelfGraph, calibrator: ScaledEmpiricalCalibrator, policy_mode: str):
        """
        حالات سیاست:
        - DECISION_EVOI: تصمیم‌محور ناب بدون پاداش برون‌یابی
        - PURE_INFO: صرفاً انقباض عدم‌قطعیت آماری
        """
        self.scm = scm
        self.calibrator = calibrator
        self.mode = policy_mode
        self.cost_factor = 0.001
        self.lambda_probe_risk = 0.5
        self.candidates = np.array([0.10, 0.20, 0.35, 0.50, 0.65, 0.75])
        self.eval_shifts = np.array([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
        self.eval_weights = norm.pdf(self.eval_shifts)
        self.eval_weights /= np.sum(self.eval_weights)

    def _evaluate_max_expected_utility(self, mu_star: float, sigma_star: float) -> float:
        sigma_star = max(sigma_star, 1e-6)
        samples = mu_star + sigma_star * self.eval_shifts
        action_expected_utils = []
        for a in DecoupledUtility.ALL_ACTIONS:
            eu_a = sum(w * DecoupledUtility.get_utility(a, s) for w, s in zip(self.eval_weights, samples))
            action_expected_utils.append(eu_a)
        return float(max(action_expected_utils))

    def evaluate_step(
        self, target: TargetSpec, ctx_str: str, executed_doses: List[float]
    ) -> Tuple[float, bool, str, str, Dict[str, float]]:
        self.scm.init_node(target.key, ctx_str)
        node = self.scm.component_models[target.key][extract_context_family(ctx_str)]

        pred_mu, sig_pred, sig_epi, _ = self.scm.predict([target], ctx_str)
        beliefs = self.calibrator.predict_beliefs(pred_mu, sig_pred, sig_epi)

        epistemic_state = "EPISTEMICALLY_UNCERTAIN"
        if beliefs["P_hazard"] >= 0.55: epistemic_state = "CONFIDENT_HAZARD"
        elif beliefs["P_gain"] >= 0.55: epistemic_state = "CONFIDENT_GAIN"
        elif beliefs["P_zero"] >= 0.55: epistemic_state = "CONFIDENT_ZERO_GAIN"

        # توقف قطعی در صورت احراز اطمینان پس از حداقل ۱ پروب
        if len(executed_doses) >= 1 and epistemic_state != "EPISTEMICALLY_UNCERTAIN":
            return 0.0, True, epistemic_state, f"STOP_{epistemic_state}", beliefs

        current_v = self._evaluate_max_expected_utility(pred_mu, sig_pred)
        best_net_score = -float("inf")
        best_dose = self.candidates[0]

        pts, wts = np.polynomial.hermite.hermgauss(5)

        for d in self.candidates:
            if any(abs(d - ed) < 0.05 for ed in executed_doses): continue

            # احتیاط ایمنی: پرهیز از دوزهای عمیق در صورت مشاهده سیگنال منفی
            if len(executed_doses) >= 1 and pred_mu < -0.05 and d > 0.40:
                continue

            x_d = np.array([d, d**2])
            pred_y = float(x_d @ node["mu"])
            std_y = np.sqrt(float(x_d @ node["cov"] @ x_d) + node["aleatoric_var"])

            prec_sim = node["precision"] + (1.0 / node["aleatoric_var"]) * np.outer(x_d, x_d)
            cov_sim = np.linalg.inv(prec_sim)
            x_target = np.array([1.0, 1.0])
            sig_pred_sim = float(np.sqrt(x_target @ cov_sim @ x_target + node["aleatoric_var"]))

            expected_posterior_v = 0.0
            probe_harm_expected = 0.0
            for pt, w in zip(pts, wts):
                y_sample = pred_y + np.sqrt(2.0) * std_y * pt
                mu_sim = cov_sim @ (node["precision"] @ node["mu"] + (1.0 / node["aleatoric_var"]) * x_d * y_sample)
                mu_full_sim = float(x_target @ mu_sim)
                expected_posterior_v += (w / np.sqrt(np.pi)) * self._evaluate_max_expected_utility(mu_full_sim, sig_pred_sim)
                probe_harm_expected += (w / np.sqrt(np.pi)) * max(0.0, -y_sample)

            if self.mode == "PURE_INFO":
                # انقباض خالص واریانس آماری بدون ترم مطلوبیت تصمیم
                info_gain = max(0.0, sig_pred - sig_pred_sim)
                net_score = info_gain - (self.cost_factor * float(d**2)) - (self.lambda_probe_risk * probe_harm_expected)
            else:
                # DECISION_EVOI: تصمیم‌محور ناب (فاقد هرگونه پاداش برون‌یابی)
                evoi_decision = max(0.0, expected_posterior_v - current_v)
                net_score = evoi_decision - (self.cost_factor * float(d**2)) - (self.lambda_probe_risk * probe_harm_expected)

            if net_score > best_net_score:
                best_net_score = net_score
                best_dose = float(d)

        if len(executed_doses) >= 3 and best_net_score <= 0.0:
            return 0.0, True, epistemic_state, "STOP_UNECONOMIC", beliefs

        return best_dose, False, epistemic_state, "CONTINUE_PROBING", beliefs


# ==============================================================================
# 6. PROTOCOL BENCHMARK CONTROLLER (MILESTONE 17.17)
# ==============================================================================

def run_milestone_17_17():
    sensors = MechanisticSensorInterface()

    corpus = {
        "ood_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ],
        "ood_distractor_reasoning": [
            PromptItem("While Rome has the Colosseum, the Eiffel Tower stands in", " Paris", " Rome"),
            PromptItem("Even though Madrid is famous, the capital of Germany is", " Berlin", " Madrid"),
            PromptItem("Egypt has pyramids, but the Eiffel Tower was built in the city of", " Paris", " Cairo"),
        ],
        "train_factual_cap": [
            PromptItem("The capital of France is the city of", " Paris", " Rome"),
            PromptItem("The capital of Japan is the bustling city of", " Tokyo", " Kyoto"),
            PromptItem("The official capital city of Germany is", " Berlin", " Munich"),
            PromptItem("The historic capital of Italy is the city of", " Rome", " Milan"),
        ]
    }

    # استخر کالیبراسیون تجربی (۱۸ نمونه؛ تفکیک لایه‌ای مطلق)
    calibration_pool_18 = [
        {"desc": "L18 MLP", "target": TargetSpec(18, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L19 MLP", "target": TargetSpec(19, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L21 MLP", "target": TargetSpec(21, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L23 Head 0", "target": TargetSpec(23, "head", 0, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L18 Head 1", "target": TargetSpec(18, "head", 1, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L19 Head 2", "target": TargetSpec(19, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L1 Head 0", "target": TargetSpec(1, "head", 0, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L8 Head 4", "target": TargetSpec(8, "head", 4, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L5 Head 2", "target": TargetSpec(5, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L5 Head 0", "target": TargetSpec(5, "head", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L8 Head 1", "target": TargetSpec(8, "head", 1, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L1 Head 2", "target": TargetSpec(1, "head", 2, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L16 MLP", "target": TargetSpec(16, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L22 Head 0", "target": TargetSpec(22, "head", 0, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L23 Head 1", "target": TargetSpec(23, "head", 1, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L21 Head 2", "target": TargetSpec(21, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L16 Head 0", "target": TargetSpec(16, "head", 0, keep_ratio=0.0), "ctx": "ood_distractor_reasoning"},
        {"desc": "L22 Head 2", "target": TargetSpec(22, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
    ]

    # مجموعه ارزیابی Held-Out کاملاً مجزا (لایه‌های کاملاً مستقل)
    final_disjoint_held_out = [
        {"desc": "HeldOut Deep Hazard L20MLP", "target": TargetSpec(20, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "HeldOut Medium Hazard L12MLP", "target": TargetSpec(12, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "HeldOut Boundary Head L12H3", "target": TargetSpec(12, "head", 3, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "HeldOut Neutral L2H1", "target": TargetSpec(2, "head", 1, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "HeldOut Distractor Gain L4H0", "target": TargetSpec(4, "head", 0, keep_ratio=0.0), "ctx": "ood_distractor_reasoning"},
        {"desc": "HeldOut Useful MLP L14MLP", "target": TargetSpec(14, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
    ]

    print("\n" + "=" * 195)
    print("🔬 MILESTONE 17.17: CAUSAL ISOLATION & DECISION-AWARE PROBING AUDIT")
    print("   Protocol: Extrapolation Heuristic Removed | Fair Matched Random | Fixed Metric Beta=0.5")
    print("=" * 195)

    # فاز ۱: آموزش کالیبراتور تجربی
    print("\n📦 Training Scaled Empirical Calibrator on 18 Disjoint Samples...")
    calibrator = ScaledEmpiricalCalibrator()
    for item in calibration_pool_18:
        target = item["target"]
        ctx = item["ctx"]
        items = corpus[ctx]
        scm = PureEpistemicSelfGraph(prior_variance=0.5, base_aleatoric=0.005)
        for d in [0.15, 0.35, 0.60]:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe], noise_sigma=0.005)
            scm.update(target.key, ctx, d, obs)
        pred_mu, sig_pred, sig_epi, _ = scm.predict([target], ctx)
        true_delta = sensors.measure_grounded_outcomes(items, targets=[target], noise_sigma=0.0)
        calibrator.store_sample(pred_mu, sig_pred, sig_epi, true_delta)

    calibrator.train_calibrator()
    print(f"📦 Calibration Status: {'✅ SUCCESS (18 Samples Fitted)' if calibrator.is_fitted else '⚠️ FALLBACK'}")

    def resolve_governor(mu: float, sigma_pred: float, beliefs: Dict[str, float]) -> str:
        if beliefs["P_hazard"] >= 0.50: return "VETO_HAZARD"
        if beliefs["P_gain"] >= 0.50: return "APPROVED"
        if beliefs["P_zero"] >= 0.50: return "VETO_ZERO_GAIN"
        if sigma_pred > 0.65 or max(beliefs.values()) < 0.40: return "VETO_EPISTEMIC"
        best_class = max(beliefs, key=beliefs.get)
        if best_class == "P_gain": return "APPROVED"
        if best_class == "P_hazard": return "VETO_HAZARD"
        return "VETO_ZERO_GAIN"

    policies = ["DECISION_EVOI", "PURE_INFO", "MATCHED_RANDOM", "NO_PROBE"]
    results_matrix = {p: {"regret": 0.0, "risk": 0.0, "final_sigma": [], "miw": [], "olr_count": 0, "sfr_count": 0, "total_steps": 0} for p in policies}

    print("\n📦 Executing Causal Isolation across Held-Out Targets (Noise=0.005 Operational)...")
    op_noise = 0.005

    for item in final_disjoint_held_out:
        target = item["target"]
        ctx = item["ctx"]
        items = corpus[ctx]
        gt_delta = sensors.measure_grounded_outcomes(items, targets=[target], noise_sigma=0.0)
        oracle_action, oracle_u = DecoupledUtility.get_omniscient_oracle(gt_delta)

        for pol in policies:
            scm = PureEpistemicSelfGraph(prior_variance=0.5, base_aleatoric=0.005)
            incurred_risk = 0.0
            executed_doses = []

            if pol == "NO_PROBE":
                pass

            elif pol == "MATCHED_RANDOM":
                # کنترل منصفانه: ۳ پروب کاملاً تصادفی در همان دامنه مجاز (0.10 تا 0.75)
                np.random.seed(SEED + hash(target.key) % 1000)
                fair_random_doses = sorted(np.random.uniform(0.10, 0.75, size=3).tolist())
                for rd in fair_random_doses:
                    probe_r = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - rd)
                    obs_r = sensors.measure_grounded_outcomes(items, targets=[probe_r], noise_sigma=op_noise)
                    incurred_risk += max(0.0, -obs_r)
                    scm.update(target.key, ctx, rd, obs_r)
                    executed_doses.append(rd)

            else:
                engine = IsolatedPolicyEngine(scm, calibrator, policy_mode=pol)
                for _ in range(4):
                    d, should_stop, ep_state, reason, beliefs = engine.evaluate_step(target, ctx, executed_doses)
                    if should_stop: break
                    probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                    obs = sensors.measure_grounded_outcomes(items, targets=[probe], noise_sigma=op_noise)
                    incurred_risk += max(0.0, -obs)
                    scm.update(target.key, ctx, d, obs)
                    executed_doses.append(d)

            pred_mu, sig_pred, sig_epi, _ = scm.predict([target], ctx)
            beliefs = calibrator.predict_beliefs(pred_mu, sig_pred, sig_epi)
            verdict = resolve_governor(pred_mu, sig_pred, beliefs)
            u = DecoupledUtility.get_utility(verdict, gt_delta)
            regret = oracle_u - u

            results_matrix[pol]["regret"] += regret
            results_matrix[pol]["risk"] += incurred_risk
            results_matrix[pol]["final_sigma"].append(sig_pred)
            results_matrix[pol]["miw"].append(3.92 * sig_pred)
            results_matrix[pol]["total_steps"] += len(executed_doses)

            if oracle_action == "VETO_HAZARD" and verdict == "APPROVED":
                results_matrix[pol]["sfr_count"] += 1
            if oracle_action == "APPROVED" and "VETO" in verdict:
                results_matrix[pol]["olr_count"] += 1

    # ==============================================================================
    # 7. ABLATION & LOSS AUDIT REPORT
    # ==============================================================================
    n_targets = len(final_disjoint_held_out)
    n_gain = sum(1 for item in final_disjoint_held_out if DecoupledUtility.get_omniscient_oracle(sensors.measure_grounded_outcomes(corpus[item["ctx"]], targets=[item["target"]], noise_sigma=0.0))[0] == "APPROVED")
    n_haz = sum(1 for item in final_disjoint_held_out if DecoupledUtility.get_omniscient_oracle(sensors.measure_grounded_outcomes(corpus[item["ctx"]], targets=[item["target"]], noise_sigma=0.0))[0] == "VETO_HAZARD")

    print("\n" + "=" * 150)
    print("📊 MILESTONE 17.17: CAUSALLY ISOLATED DECISION-AWARE AUDIT REPORT")
    print("-" * 150)
    print(f"{'Policy Architecture':<18} | {'Mean Final σ':<13} | {'Mean MIW':<10} | {'Total Risk':<12} | {'Total Regret':<13} | {'Risk-Adj Loss (β=0.5)':<23} | {'OLR (%)':<9} | {'SFR (%)'}")
    print("-" * 150)

    beta = 0.5
    for pol in policies:
        m_sig = float(np.mean(results_matrix[pol]["final_sigma"]))
        m_miw = float(np.mean(results_matrix[pol]["miw"]))
        t_risk = results_matrix[pol]["risk"]
        t_reg = results_matrix[pol]["regret"]
        risk_adj_loss = t_reg + (beta * t_risk)
        olr = (results_matrix[pol]["olr_count"] / max(1, n_gain)) * 100.0
        sfr = (results_matrix[pol]["sfr_count"] / max(1, n_haz)) * 100.0

        print(f"{pol:<18} | {m_sig:<13.4f} | {m_miw:<10.3f} | {t_risk:<12.4f} | {t_reg:<13.4f} | {risk_adj_loss:<23.4f} | {olr:<9.1f} | {sfr:<7.1f}")

    print("=" * 150 + "\n")

    with open("m17_17_isolated_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(results_matrix), f, indent=2)
    print("📁 Telemetry saved to 'm17_17_isolated_results.json'.")


if __name__ == "__main__":
    run_milestone_17_17()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 17.17: CAUSAL ISOLATION & DECISION-AWARE PROBING AUDIT
   Protocol: Extrapolation Heuristic Removed | Fair Matched Random | Fixed Metric Beta=0.5

📦 Training Scaled Empirical Calibrator on 18 Disjoint Samples...
📦 Calibration Status: ✅ SUCCESS (18 Samples Fitted)

📦 Executing Causal Isolation across Held-Out Targets (Noise=0.005 Operational)...

📊 MILESTONE 17.17: CAUSALLY ISOLATED DECISION-AWARE AUDIT REPORT
------------------------------------------------------------------------------------------------------------------------------------------------------
Policy Architecture | Mean Final σ  | Mean MIW   | Total Risk   | Total Regret  | Risk-Adj Loss (β=0.5)   | OLR (%)   | SFR (%)
------------------------------------------------------------------------------------------------------------------------------------------------------
DECISION_EVOI      | 0.2779        | 1.089      | 0.3553       | 0.0250        

In [1]:
# ==============================================================================
# MILESTONE 18A: CROSS-CONTEXT MECHANISM TRANSFER FEASIBILITY AUDIT
# ==============================================================================
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ------------------------------------------------------------------------------
# 0. SERIALIZATION & HELPERS
# ------------------------------------------------------------------------------
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, torch.Tensor): return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None: return obj
    if hasattr(obj, "__dataclass_fields__"): return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_", "target_"]:
        if ctx_str.startswith(prefix): return ctx_str[len(prefix):]
    return ctx_str


# ------------------------------------------------------------------------------
# 1. UTILITY MATRIX & GOVERNOR
# ------------------------------------------------------------------------------
class DecoupledUtility:
    THRESH_GAIN = 0.005
    THRESH_HAZARD = -0.05
    ALL_ACTIONS = ["APPROVED", "VETO_ZERO_GAIN", "VETO_HAZARD", "VETO_EPISTEMIC"]

    @staticmethod
    def get_utility(action: str, delta_m: float) -> float:
        if delta_m >= DecoupledUtility.THRESH_GAIN:
            if action == "APPROVED": return float(delta_m)
            if action == "VETO_EPISTEMIC": return -0.005
            if action == "VETO_ZERO_GAIN": return -0.015
            if action == "VETO_HAZARD": return -0.03
        elif delta_m < DecoupledUtility.THRESH_HAZARD:
            if action == "VETO_HAZARD": return 0.05
            if action == "VETO_EPISTEMIC": return 0.03
            if action == "VETO_ZERO_GAIN": return 0.01
            if action == "APPROVED": return float(10.0 * delta_m)
        else:
            if action == "VETO_ZERO_GAIN": return 0.005
            if action == "VETO_EPISTEMIC": return 0.002
            if action == "VETO_HAZARD": return 0.0
            if action == "APPROVED": return float(delta_m)
        return -1.0

    @staticmethod
    def get_omniscient_oracle(delta_m: float) -> Tuple[str, float]:
        utils = [DecoupledUtility.get_utility(a, delta_m) for a in DecoupledUtility.ALL_ACTIONS]
        best_idx = int(np.argmax(utils))
        return DecoupledUtility.ALL_ACTIONS[best_idx], utils[best_idx]


def compute_belief_distribution(mu: float, sig_pred: float) -> Dict[str, float]:
    sig_pred = max(sig_pred, 1e-6)
    p_haz = float(norm.cdf((DecoupledUtility.THRESH_HAZARD - mu) / sig_pred))
    p_gain = float(1.0 - norm.cdf((DecoupledUtility.THRESH_GAIN - mu) / sig_pred))
    p_zero = max(0.0, 1.0 - p_haz - p_gain)
    return {"P_gain": p_gain, "P_zero": p_zero, "P_hazard": p_haz}


def resolve_governor(mu: float, sigma_pred: float, beliefs: Dict[str, float]) -> str:
    if beliefs["P_hazard"] >= 0.50: return "VETO_HAZARD"
    if beliefs["P_gain"] >= 0.50: return "APPROVED"
    if beliefs["P_zero"] >= 0.50: return "VETO_ZERO_GAIN"
    if sigma_pred > 0.65 or max(beliefs.values()) < 0.40: return "VETO_EPISTEMIC"
    best_class = max(beliefs, key=beliefs.get)
    if best_class == "P_gain": return "APPROVED"
    if best_class == "P_hazard": return "VETO_HAZARD"
    return "VETO_ZERO_GAIN"


# ------------------------------------------------------------------------------
# 2. MECHANISTIC SENSORS INTERFACE
# ------------------------------------------------------------------------------
@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(
        self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None, noise_sigma: float = 0.0
    ) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets: return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        raw_delta = float(torch.mean(int_m - clean_m).item())

        if noise_sigma > 0.0:
            return raw_delta + float(np.random.normal(0.0, noise_sigma))
        return raw_delta


# ------------------------------------------------------------------------------
# 3. TRANSFERABLE BAYESIAN SELF-GRAPH
# ------------------------------------------------------------------------------
class TransferableBayesianSelfGraph:
    def __init__(self, default_prior_variance: float = 0.5, base_aleatoric: float = 0.005):
        self.default_prior_variance = default_prior_variance
        self.base_aleatoric = base_aleatoric
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}

    def init_node(
        self,
        target_key: str,
        ctx_str: str,
        initial_mu: Optional[np.ndarray] = None,
        initial_cov: Optional[np.ndarray] = None,
        inflation_factor: float = 1.0
    ):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            if initial_mu is not None and initial_cov is not None:
                mu_init = np.array(initial_mu, dtype=float)
                cov_init = np.array(initial_cov, dtype=float) * inflation_factor
                prec_init = np.linalg.inv(cov_init)
            else:
                mu_init = np.zeros(2)
                cov_init = np.eye(2) * self.default_prior_variance
                prec_init = np.eye(2) / self.default_prior_variance

            self.component_models[target_key][family] = {
                "mu": mu_init,
                "precision": prec_init,
                "cov": cov_init,
                "aleatoric_var": self.base_aleatoric,
                "history": [],
                "n": 0
            }

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float):
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        node["history"].append((dose, delta_obs))

        if len(node["history"]) >= 3:
            y_vals = np.array([h[1] for h in node["history"]])
            diffs = np.diff(y_vals)
            node["aleatoric_var"] = max(self.base_aleatoric, float(0.5 * np.mean(diffs**2)))

        inv_noise = 1.0 / max(node["aleatoric_var"], 1e-6)
        prec_old = node["precision"]
        mu_old = node["mu"]

        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float, float, float]:
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        var_epi = 0.0
        var_ale = 0.0

        for t in targets:
            if t.dose == 0.0: continue
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            x_target = np.array([t.dose, t.dose**2])
            pred_delta += float(x_target @ node["mu"])
            var_epi += float(x_target @ node["cov"] @ x_target)
            var_ale += float(node["aleatoric_var"])

        total_var = var_epi + var_ale
        return pred_delta, float(np.sqrt(total_var)), float(np.sqrt(var_epi)), float(np.sqrt(var_ale))

    def get_posterior_distribution(self, target_key: str, ctx_str: str) -> Tuple[np.ndarray, np.ndarray]:
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]
        return np.copy(node["mu"]), np.copy(node["cov"])


# ------------------------------------------------------------------------------
# 4. BENCHMARK EXECUTION
# ------------------------------------------------------------------------------
def run_milestone_18a():
    sensors = MechanisticSensorInterface()

    corpus = {
        "train_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ],
        "target_distractor_reasoning": [
            PromptItem("While Rome has the Colosseum, the Eiffel Tower stands in", " Paris", " Rome"),
            PromptItem("Even though Madrid is famous, the capital of Germany is", " Berlin", " Madrid"),
            PromptItem("Egypt has pyramids, but the Eiffel Tower was built in the city of", " Paris", " Cairo"),
        ],
        "target_factual_cap": [
            PromptItem("The capital of France is the city of", " Paris", " Rome"),
            PromptItem("The capital of Japan is the bustling city of", " Tokyo", " Kyoto"),
            PromptItem("The official capital city of Germany is", " Berlin", " Munich"),
            PromptItem("The historic capital of Italy is the city of", " Rome", " Milan"),
        ]
    }

    source_context = "train_syntactic_agreement"

    audit_components = [
        {"desc": "Upstream Head L2H1", "target": TargetSpec(2, "head", 1, keep_ratio=0.0)},
        {"desc": "Mid Syntactic Head L8H4", "target": TargetSpec(8, "head", 4, keep_ratio=0.0)},
        {"desc": "Late Reasoning Head L22H0", "target": TargetSpec(22, "head", 0, keep_ratio=0.0)},
        {"desc": "Critical Negative Head L12H3", "target": TargetSpec(12, "head", 3, keep_ratio=0.0)},
        {"desc": "Deep Negative MLP L20MLP", "target": TargetSpec(20, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Mid Hazard MLP L12MLP", "target": TargetSpec(12, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Positive Mid MLP L16MLP", "target": TargetSpec(16, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Late Head L21H2", "target": TargetSpec(21, "head", 2, keep_ratio=0.0)},
    ]

    target_contexts = [
        {"name": "Context_B (Distractor Reasoning)", "ctx": "target_distractor_reasoning"},
        {"name": "Context_C (Factual Capitals)", "ctx": "target_factual_cap"}
    ]

    print("\n" + "=" * 170)
    print("🔬 MILESTONE 18A: MECHANISM TRANSFER FEASIBILITY AUDIT")
    print("   Evaluating: Does learning in Context A (Syntactic) transfer to Context B (Distractor) and Context C (Factual)?")
    print("   Testing for: Zero-Shot Transfer Error | Positive vs Negative Transfer | Inflation Damping")
    print("=" * 170)

    # --------------------------------------------------------------------------
    # PHASE 1: Learn Source Priors in Context A
    # --------------------------------------------------------------------------
    print(f"\n📦 PHASE 1: Training Source Priors in '{source_context}'...")
    source_scm = TransferableBayesianSelfGraph(default_prior_variance=0.5, base_aleatoric=0.005)
    source_items = corpus[source_context]
    source_profiles = {}

    for comp in audit_components:
        target = comp["target"]
        desc = comp["desc"]
        for d in [0.15, 0.35, 0.60]:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(source_items, targets=[probe], noise_sigma=0.005)
            source_scm.update(target.key, source_context, d, obs)

        pred_mu, sig_pred, sig_epi, _ = source_scm.predict([target], source_context)
        gt_source = sensors.measure_grounded_outcomes(source_items, targets=[target], noise_sigma=0.0)
        mu_post, cov_post = source_scm.get_posterior_distribution(target.key, source_context)
        source_profiles[target.key] = {
            "mu": mu_post,
            "cov": cov_post,
            "pred_mu": pred_mu,
            "gt_source": gt_source
        }
        print(f"  • Learned Source [{target.key:<10} | {desc:<30}]: μ_eff = {pred_mu:+7.4f} | GT = {gt_source:+7.4f} | Tr(Σ) = {np.trace(cov_post):.5f}")

    # --------------------------------------------------------------------------
    # PHASE 2: Zero-Shot Transfer Evaluation on Target Contexts
    # --------------------------------------------------------------------------
    print("\n📦 PHASE 2: Cross-Context Transfer Evaluation (Comparing 3 Regimes per Target)...")
    records = []

    for t_ctx_info in target_contexts:
        t_ctx_name = t_ctx_info["name"]
        t_ctx = t_ctx_info["ctx"]
        t_items = corpus[t_ctx]
        print(f"\n{'-'*170}")
        print(f"🎯 TARGET CONTEXT: {t_ctx_name} ({t_ctx})")
        print(f"{'-'*170}")
        print(f"{'Component':<12} | {'GT Target':<10} | {'Regime':<17} | {'Pred μ_0':<9} | {'σ_pred_0':<9} | {'Prior Error':<12} | {'Verdict_0':<14} | {'Regret_0':<9} | {'Transfer Type'}")
        print(f"{'-'*170}")

        for comp in audit_components:
            target = comp["target"]
            gt_target = sensors.measure_grounded_outcomes(t_items, targets=[target], noise_sigma=0.0)
            oracle_action, oracle_u = DecoupledUtility.get_omniscient_oracle(gt_target)

            src_mu = source_profiles[target.key]["mu"]
            src_cov = source_profiles[target.key]["cov"]

            regimes = [
                ("RAW_PRIOR", None, None, 1.0),
                ("DIRECT_TRANSFER", src_mu, src_cov, 1.0),
                ("DAMPED_TRANSFER", src_mu, src_cov, 2.5)
            ]

            comp_records = {}

            for r_name, init_mu, init_cov, inflation in regimes:
                scm = TransferableBayesianSelfGraph(default_prior_variance=0.5, base_aleatoric=0.005)
                scm.init_node(target.key, t_ctx, initial_mu=init_mu, initial_cov=init_cov, inflation_factor=inflation)

                p_mu_0, s_pred_0, s_epi_0, s_ale_0 = scm.predict([target], t_ctx)
                beliefs_0 = compute_belief_distribution(p_mu_0, s_pred_0)
                verdict_0 = resolve_governor(p_mu_0, s_pred_0, beliefs_0)
                u_0 = DecoupledUtility.get_utility(verdict_0, gt_target)
                regret_0 = oracle_u - u_0
                prior_error = abs(p_mu_0 - gt_target)

                comp_records[r_name] = {
                    "pred_mu": p_mu_0,
                    "sigma": s_pred_0,
                    "error": prior_error,
                    "regret": regret_0,
                    "verdict": verdict_0,
                    "oracle": oracle_action
                }

            raw_err = comp_records["RAW_PRIOR"]["error"]
            dir_err = comp_records["DIRECT_TRANSFER"]["error"]

            if dir_err < raw_err - 0.02:
                transfer_type = "🟢 POSITIVE"
            elif dir_err > raw_err + 0.02:
                transfer_type = "🔴 NEGATIVE"
            else:
                transfer_type = "⚪ NEUTRAL"

            for r_name in ["RAW_PRIOR", "DIRECT_TRANSFER", "DAMPED_TRANSFER"]:
                rec = comp_records[r_name]
                t_label = transfer_type if r_name == "DIRECT_TRANSFER" else ("🛡️ DAMPED" if r_name == "DAMPED_TRANSFER" else "-")
                print(f"{target.key:<12} | {gt_target:+10.4f} | {r_name:<17} | {rec['pred_mu']:+9.4f} | {rec['sigma']:9.4f} | {rec['error']:12.4f} | {rec['verdict']:<14} | {rec['regret']:9.4f} | {t_label}")

                records.append({
                    "target_context": t_ctx,
                    "target_key": target.key,
                    "gt_target": gt_target,
                    "oracle": oracle_action,
                    "regime": r_name,
                    "pred_mu_0": rec["pred_mu"],
                    "sig_pred_0": rec["sigma"],
                    "prior_error": rec["error"],
                    "verdict_0": rec["verdict"],
                    "regret_0": rec["regret"],
                    "transfer_type": transfer_type
                })
            print(f"{'-'*170}")

    # --------------------------------------------------------------------------
    # PHASE 3: Macro Audit & Transfer Feasibility Matrix
    # --------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 MACRO TRANSFER FEASIBILITY SUMMARY:")
    print("-" * 140)
    print(f"{'Target Context':<32} | {'Regime':<17} | {'Mean Prior Error':<18} | {'Mean σ_0':<10} | {'Total Regret_0':<15} | {'OLR_0 (%)':<10} | {'SFR_0 (%)'}")
    print("-" * 140)

    for t_ctx_info in target_contexts:
        t_ctx = t_ctx_info["ctx"]
        sub = [r for r in records if r["target_context"] == t_ctx]
        for r_name in ["RAW_PRIOR", "DIRECT_TRANSFER", "DAMPED_TRANSFER"]:
            r_sub = [r for r in sub if r["regime"] == r_name]
            mean_err = np.mean([r["prior_error"] for r in r_sub])
            mean_sig = np.mean([r["sig_pred_0"] for r in r_sub])
            tot_reg = np.sum([r["regret_0"] for r in r_sub])

            n_gain = sum(1 for r in r_sub if r["oracle"] == "APPROVED")
            n_haz = sum(1 for r in r_sub if r["oracle"] == "VETO_HAZARD")

            olr = (sum(1 for r in r_sub if r["oracle"] == "APPROVED" and "VETO" in r["verdict_0"]) / max(1, n_gain)) * 100.0
            sfr = (sum(1 for r in r_sub if r["oracle"] == "VETO_HAZARD" and r["verdict_0"] == "APPROVED") / max(1, n_haz)) * 100.0

            print(f"{t_ctx:<32} | {r_name:<17} | {mean_err:<18.4f} | {mean_sig:<10.4f} | {tot_reg:<15.4f} | {olr:<10.1f} | {sfr:<7.1f}")
        print("-" * 140)

    print("\n" + "=" * 140)
    print("🔍 TRANSFER TYPE DISTRIBUTION (Direct Transfer vs Raw Prior):")
    print("-" * 140)
    direct_records = [r for r in records if r["regime"] == "DIRECT_TRANSFER"]
    pos_count = sum(1 for r in direct_records if "POSITIVE" in r["transfer_type"])
    neg_count = sum(1 for r in direct_records if "NEGATIVE" in r["transfer_type"])
    neu_count = sum(1 for r in direct_records if "NEUTRAL" in r["transfer_type"])
    tot_evals = len(direct_records)
    print(f"  • 🟢 Positive Transfer Nodes : {pos_count} / {tot_evals} ({pos_count/tot_evals*100:.1f}%)")
    print(f"  • 🔴 Negative Transfer Nodes : {neg_count} / {tot_evals} ({neg_count/tot_evals*100:.1f}%)")
    print(f"  • ⚪ Neutral / Stable Nodes  : {neu_count} / {tot_evals} ({neu_count/tot_evals*100:.1f}%)")
    print("=" * 140 + "\n")

    with open("m18a_transfer_feasibility_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(records), f, indent=2)
    print("📁 Transfer Feasibility Audit successfully saved to 'm18a_transfer_feasibility_results.json'.")


if __name__ == "__main__":
    run_milestone_18a()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


/tmp/ipykernel_3022/531131173.py:125: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 18A: MECHANISM TRANSFER FEASIBILITY AUDIT
   Evaluating: Does learning in Context A (Syntactic) transfer to Context B (Distractor) and Context C (Factual)?
   Testing for: Zero-Shot Transfer Error | Positive vs Negative Transfer | Inflation Damping

📦 PHASE 1: Training Source Priors in 'train_syntactic_agreement'...
  • Learned Source [L2_head_1  | Upstream Head L2H1            ]: μ_eff = -0.0109 | GT = -0.0169 | Tr(Σ) = 0.30164
  • Learned Source [L8_head_4  | Mid Syntactic Head L8H4       ]: μ_eff = +0.0008 | GT = +0.0035 | Tr(Σ) = 0.30164
  • Learned Source [L22_head_0 | Late Reasoning Head L22H0     ]: μ_eff = -0.0295 | GT = -0.0129 | Tr(Σ) = 0.30164
  • Learned Source [L12_head_3 | Critical Negative Head L12H3  ]: μ_eff = -0.0541 | GT = -0.0430 | Tr(Σ) = 0.30164
  • Learned Source [L20_mlp_0  | Deep Negative MLP L20MLP      ]: μ_eff = -0.6545 | GT = -0.5996 | Tr(Σ) = 0.32574
  • Learned Source [L12_mlp_0

In [2]:
# ==============================================================================
# MILESTONE 18B: GATED 1-PROBE TRANSFER & CAUSAL CONSISTENCY AUDIT
# ==============================================================================
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ------------------------------------------------------------------------------
# 0. SERIALIZATION & HELPERS
# ------------------------------------------------------------------------------
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, torch.Tensor): return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None: return obj
    if hasattr(obj, "__dataclass_fields__"): return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_", "target_"]:
        if ctx_str.startswith(prefix): return ctx_str[len(prefix):]
    return ctx_str


# ------------------------------------------------------------------------------
# 1. UTILITY MATRIX & GOVERNANCE
# ------------------------------------------------------------------------------
class DecoupledUtility:
    THRESH_GAIN = 0.005
    THRESH_HAZARD = -0.05
    ALL_ACTIONS = ["APPROVED", "VETO_ZERO_GAIN", "VETO_HAZARD", "VETO_EPISTEMIC"]

    @staticmethod
    def get_utility(action: str, delta_m: float) -> float:
        if delta_m >= DecoupledUtility.THRESH_GAIN:
            if action == "APPROVED": return float(delta_m)
            if action == "VETO_EPISTEMIC": return -0.005
            if action == "VETO_ZERO_GAIN": return -0.015
            if action == "VETO_HAZARD": return -0.03
        elif delta_m < DecoupledUtility.THRESH_HAZARD:
            if action == "VETO_HAZARD": return 0.05
            if action == "VETO_EPISTEMIC": return 0.03
            if action == "VETO_ZERO_GAIN": return 0.01
            if action == "APPROVED": return float(10.0 * delta_m)
        else:
            if action == "VETO_ZERO_GAIN": return 0.005
            if action == "VETO_EPISTEMIC": return 0.002
            if action == "VETO_HAZARD": return 0.0
            if action == "APPROVED": return float(delta_m)
        return -1.0

    @staticmethod
    def get_omniscient_oracle(delta_m: float) -> Tuple[str, float]:
        utils = [DecoupledUtility.get_utility(a, delta_m) for a in DecoupledUtility.ALL_ACTIONS]
        best_idx = int(np.argmax(utils))
        return DecoupledUtility.ALL_ACTIONS[best_idx], utils[best_idx]


def compute_belief_distribution(mu: float, sig_pred: float) -> Dict[str, float]:
    sig_pred = max(sig_pred, 1e-6)
    p_haz = float(norm.cdf((DecoupledUtility.THRESH_HAZARD - mu) / sig_pred))
    p_gain = float(1.0 - norm.cdf((DecoupledUtility.THRESH_GAIN - mu) / sig_pred))
    p_zero = max(0.0, 1.0 - p_haz - p_gain)
    return {"P_gain": p_gain, "P_zero": p_zero, "P_hazard": p_haz}


def resolve_governor(mu: float, sigma_pred: float, beliefs: Dict[str, float]) -> str:
    if beliefs["P_hazard"] >= 0.50: return "VETO_HAZARD"
    if beliefs["P_gain"] >= 0.50: return "APPROVED"
    if beliefs["P_zero"] >= 0.50: return "VETO_ZERO_GAIN"
    if sigma_pred > 0.65 or max(beliefs.values()) < 0.40: return "VETO_EPISTEMIC"
    best_class = max(beliefs, key=beliefs.get)
    if best_class == "P_gain": return "APPROVED"
    if best_class == "P_hazard": return "VETO_HAZARD"
    return "VETO_ZERO_GAIN"


# ------------------------------------------------------------------------------
# 2. MECHANISTIC SENSORS INTERFACE
# ------------------------------------------------------------------------------
@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(
        self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None, noise_sigma: float = 0.0
    ) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets: return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        raw_delta = float(torch.mean(int_m - clean_m).item())

        if noise_sigma > 0.0:
            return raw_delta + float(np.random.normal(0.0, noise_sigma))
        return raw_delta


# ------------------------------------------------------------------------------
# 3. GATED BAYESIAN SCM (WITH ONLINE REJECTION & FALLBACK)
# ------------------------------------------------------------------------------
class GatedBayesianSelfGraph:
    def __init__(self, default_prior_variance: float = 0.5, base_aleatoric: float = 0.005):
        self.default_prior_variance = default_prior_variance
        self.base_aleatoric = base_aleatoric
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}

    def init_node(
        self,
        target_key: str,
        ctx_str: str,
        initial_mu: Optional[np.ndarray] = None,
        initial_cov: Optional[np.ndarray] = None
    ):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            has_imported = (initial_mu is not None and initial_cov is not None)
            if has_imported:
                mu_init = np.array(initial_mu, dtype=float)
                cov_init = np.array(initial_cov, dtype=float)
                prec_init = np.linalg.inv(cov_init)
            else:
                mu_init = np.zeros(2)
                cov_init = np.eye(2) * self.default_prior_variance
                prec_init = np.eye(2) / self.default_prior_variance

            self.component_models[target_key][family] = {
                "mu": mu_init,
                "precision": prec_init,
                "cov": cov_init,
                "aleatoric_var": self.base_aleatoric,
                "history": [],
                "imported_prior": has_imported,
                "prior_status": "IMPORTED_CANDIDATE" if has_imported else "UNINFORMATIVE_COLD",
                "n": 0
            }

    def reset_to_uninformative(self, target_key: str, ctx_str: str):
        """ابطال پیشین انتقال‌یافته در صورت رد شدن توسط گیت تجانس"""
        family = extract_context_family(ctx_str)
        node = self.component_models[target_key][family]
        node["mu"] = np.zeros(2)
        node["cov"] = np.eye(2) * self.default_prior_variance
        node["precision"] = np.eye(2) / self.default_prior_variance
        node["imported_prior"] = False
        node["prior_status"] = "REJECTED_FALLBACK_RAW"

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float):
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        node["history"].append((dose, delta_obs))

        if len(node["history"]) >= 3:
            y_vals = np.array([h[1] for h in node["history"]])
            diffs = np.diff(y_vals)
            node["aleatoric_var"] = max(self.base_aleatoric, float(0.5 * np.mean(diffs**2)))

        inv_noise = 1.0 / max(node["aleatoric_var"], 1e-6)
        prec_old = node["precision"]
        mu_old = node["mu"]

        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float, float, float]:
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        var_epi = 0.0
        var_ale = 0.0

        for t in targets:
            if t.dose == 0.0: continue
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            x_target = np.array([t.dose, t.dose**2])
            pred_delta += float(x_target @ node["mu"])
            var_epi += float(x_target @ node["cov"] @ x_target)
            var_ale += float(node["aleatoric_var"])

        total_var = var_epi + var_ale
        return pred_delta, float(np.sqrt(total_var)), float(np.sqrt(var_epi)), float(np.sqrt(var_ale))

    def evaluate_consistency_gate(
        self, target_key: str, ctx_str: str, anchor_dose: float, obs_anchor: float
    ) -> Tuple[bool, str, float]:
        """
        گیت تجانس علّی: آیا پروب کوچک $d=0.15$ پیشین وارداتی را تأیید می‌کند؟
        """
        family = extract_context_family(ctx_str)
        node = self.component_models[target_key][family]
        if not node["imported_prior"]:
            return True, "NO_PRIOR_TO_CHECK", 0.0

        x_d = np.array([anchor_dose, anchor_dose**2])
        pred_anchor = float(x_d @ node["mu"])
        var_anchor = float(x_d @ node["cov"] @ x_d) + node["aleatoric_var"]
        std_anchor = max(np.sqrt(var_anchor), 1e-5)

        # محاسبه Z-Score انحراف مشاهده از پیشین
        z_score = abs(obs_anchor - pred_anchor) / std_anchor

        # شرط ۱: اگر علامت اثر متناقض و معنادار باشد (وارونگی نقش)
        sign_flip = (pred_anchor * obs_anchor < -0.002) and (abs(obs_anchor) > 0.02)
        # شرط ۲: انحراف آماری فراتر از ۲.۵ سیگما
        statistical_conflict = (z_score > 2.5)

        if sign_flip or statistical_conflict:
            node["prior_status"] = "REJECTED_NEGATIVE_TRANSFER"
            return False, "REJECT_CONFLICT", z_score

        node["prior_status"] = "ACCEPTED_CONSISTENT"
        return True, "ACCEPT_CONSISTENT", z_score


# ------------------------------------------------------------------------------
# 4. ACTIVE EVOI EXTENSION (برای گره‌های ردشده یا نیازمند کاوش)
# ------------------------------------------------------------------------------
class ActiveEVoIProber:
    def __init__(self, scm: GatedBayesianSelfGraph):
        self.scm = scm
        self.candidates = np.array([0.25, 0.40, 0.60, 0.75])
        self.cost_factor = 0.001
        self.lambda_probe_risk = 0.5
        self.eval_shifts = np.array([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
        self.eval_weights = norm.pdf(self.eval_shifts)
        self.eval_weights /= np.sum(self.eval_weights)

    def _evaluate_max_expected_utility(self, mu_star: float, sigma_star: float) -> float:
        sigma_star = max(sigma_star, 1e-6)
        samples = mu_star + sigma_star * self.eval_shifts
        action_expected_utils = []
        for a in DecoupledUtility.ALL_ACTIONS:
            eu_a = sum(w * DecoupledUtility.get_utility(a, s) for w, s in zip(self.eval_weights, samples))
            action_expected_utils.append(eu_a)
        return float(max(action_expected_utils))

    def select_next_dose(self, target: TargetSpec, ctx_str: str, executed_doses: List[float]) -> Tuple[float, bool]:
        node = self.scm.component_models[target.key][extract_context_family(ctx_str)]
        pred_mu, sig_pred, _, _ = self.scm.predict([target], ctx_str)
        current_v = self._evaluate_max_expected_utility(pred_mu, sig_pred)
        best_score = -float("inf")
        best_dose = self.candidates[0]

        pts, wts = np.polynomial.hermite.hermgauss(5)

        for d in self.candidates:
            if any(abs(d - ed) < 0.06 for ed in executed_doses): continue
            if len(executed_doses) >= 1 and pred_mu < -0.05 and d > 0.40: continue

            x_d = np.array([d, d**2])
            pred_y = float(x_d @ node["mu"])
            std_y = np.sqrt(float(x_d @ node["cov"] @ x_d) + node["aleatoric_var"])

            prec_sim = node["precision"] + (1.0 / node["aleatoric_var"]) * np.outer(x_d, x_d)
            cov_sim = np.linalg.inv(prec_sim)
            x_target = np.array([1.0, 1.0])
            sig_pred_sim = float(np.sqrt(x_target @ cov_sim @ x_target + node["aleatoric_var"]))

            expected_posterior_v = 0.0
            probe_harm_expected = 0.0
            for pt, w in zip(pts, wts):
                y_sample = pred_y + np.sqrt(2.0) * std_y * pt
                mu_sim = cov_sim @ (node["precision"] @ node["mu"] + (1.0 / node["aleatoric_var"]) * x_d * y_sample)
                mu_full_sim = float(x_target @ mu_sim)
                expected_posterior_v += (w / np.sqrt(np.pi)) * self._evaluate_max_expected_utility(mu_full_sim, sig_pred_sim)
                probe_harm_expected += (w / np.sqrt(np.pi)) * max(0.0, -y_sample)

            evoi_decision = max(0.0, expected_posterior_v - current_v)
            net_score = evoi_decision - (self.cost_factor * float(d**2)) - (self.lambda_probe_risk * probe_harm_expected)

            if net_score > best_score:
                best_score = net_score
                best_dose = float(d)

        if best_score <= 0.0 and len(executed_doses) >= 2:
            return 0.0, True
        return best_dose, False


# ------------------------------------------------------------------------------
# 5. BENCHMARK EXECUTION (MILESTONE 18B)
# ------------------------------------------------------------------------------
def run_milestone_18b():
    sensors = MechanisticSensorInterface()

    corpus = {
        "train_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ],
        "target_distractor_reasoning": [
            PromptItem("While Rome has the Colosseum, the Eiffel Tower stands in", " Paris", " Rome"),
            PromptItem("Even though Madrid is famous, the capital of Germany is", " Berlin", " Madrid"),
            PromptItem("Egypt has pyramids, but the Eiffel Tower was built in the city of", " Paris", " Cairo"),
        ],
        "target_factual_cap": [
            PromptItem("The capital of France is the city of", " Paris", " Rome"),
            PromptItem("The capital of Japan is the bustling city of", " Tokyo", " Kyoto"),
            PromptItem("The official capital city of Germany is", " Berlin", " Munich"),
            PromptItem("The historic capital of Italy is the city of", " Rome", " Milan"),
        ]
    }

    source_context = "train_syntactic_agreement"

    audit_components = [
        {"desc": "Upstream Head L2H1", "target": TargetSpec(2, "head", 1, keep_ratio=0.0)},
        {"desc": "Mid Syntactic Head L8H4", "target": TargetSpec(8, "head", 4, keep_ratio=0.0)},
        {"desc": "Late Reasoning Head L22H0", "target": TargetSpec(22, "head", 0, keep_ratio=0.0)},
        {"desc": "Critical Negative Head L12H3", "target": TargetSpec(12, "head", 3, keep_ratio=0.0)},
        {"desc": "Deep Negative MLP L20MLP", "target": TargetSpec(20, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Mid Hazard MLP L12MLP", "target": TargetSpec(12, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Positive Mid MLP L16MLP", "target": TargetSpec(16, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Late Head L21H2", "target": TargetSpec(21, "head", 2, keep_ratio=0.0)},
    ]

    target_contexts = [
        {"name": "Context_B (Distractor Reasoning)", "ctx": "target_distractor_reasoning"},
        {"name": "Context_C (Factual Capitals)", "ctx": "target_factual_cap"}
    ]

    print("\n" + "=" * 185)
    print("🔬 MILESTONE 18B: GATED 1-PROBE TRANSFER & CAUSAL CONSISTENCY AUDIT")
    print("   Evaluating: Safe 1-Probe Anchor -> Causal Consistency Gate -> Dynamic Negative-Transfer Rejection")
    print("=" * 185)

    # --------------------------------------------------------------------------
    # فاز ۱: استخراج پیشین‌ها از کانتکست مبدأ
    # --------------------------------------------------------------------------
    print(f"\n📦 PHASE 1: Harvesting Source Priors in '{source_context}'...")
    source_scm = GatedBayesianSelfGraph(default_prior_variance=0.5, base_aleatoric=0.005)
    source_items = corpus[source_context]
    source_profiles = {}

    for comp in audit_components:
        target = comp["target"]
        for d in [0.15, 0.35, 0.60]:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(source_items, targets=[probe], noise_sigma=0.005)
            source_scm.update(target.key, source_context, d, obs)

        pred_mu, sig_pred, _, _ = source_scm.predict([target], source_context)
        gt_source = sensors.measure_grounded_outcomes(source_items, targets=[target], noise_sigma=0.0)
        node = source_scm.component_models[target.key][extract_context_family(source_context)]
        source_profiles[target.key] = {
            "mu": np.copy(node["mu"]),
            "cov": np.copy(node["cov"]),
            "pred_mu": pred_mu,
            "gt_source": gt_source
        }
        print(f"  • Source Profile [{target.key:<10}]: μ = {pred_mu:+7.4f} | GT = {gt_source:+7.4f}")

    # --------------------------------------------------------------------------
    # فاز ۲ و ۳: آزمون گیت تجانس در کانتکست‌های مقصد
    # --------------------------------------------------------------------------
    records = []
    anchor_dose = 0.15

    for t_ctx_info in target_contexts:
        t_ctx_name = t_ctx_info["name"]
        t_ctx = t_ctx_info["ctx"]
        t_items = corpus[t_ctx]

        print(f"\n{'-'*185}")
        print(f"🎯 EVALUATION IN TARGET: {t_ctx_name} ({t_ctx})")
        print(f"{'-'*185}")
        print(f"{'Component':<11} | {'GT':<7} | {'Regime':<16} | {'Gate Decision':<20} | {'Z-Score':<7} | {'Probes':<6} | {'Risk':<7} | {'Verdict':<14} | {'Regret':<7} | {'Status'}")
        print(f"{'-'*185}")

        for comp in audit_components:
            target = comp["target"]
            gt_target = sensors.measure_grounded_outcomes(t_items, targets=[target], noise_sigma=0.0)
            oracle_action, oracle_u = DecoupledUtility.get_omniscient_oracle(gt_target)
            src_mu = source_profiles[target.key]["mu"]
            src_cov = source_profiles[target.key]["cov"]

            # مقایسه ۳ استراتژی:
            # ۱. DIRECT_TRANSFER (روش آسیب‌پذیر قبلی M18A)
            # ۲. RAW_EXPLORATION (کاوش مستقل بدون پیشین)
            # ۳. GATED_1PROBE_TRANSFER (معماری جدید M18B)
            regimes = ["DIRECT_M18A", "RAW_COLD", "GATED_1PROBE"]

            for reg in regimes:
                scm = GatedBayesianSelfGraph(default_prior_variance=0.5, base_aleatoric=0.005)
                executed_doses = []
                incurred_risk = 0.0
                gate_decision = "BYPASSED"
                z_metric = 0.0

                if reg == "DIRECT_M18A":
                    # انتقال مستقیم بدون هیچ پروبینگ
                    scm.init_node(target.key, t_ctx, initial_mu=src_mu, initial_cov=src_cov)
                    p_mu, s_pred, _, _ = scm.predict([target], t_ctx)

                elif reg == "RAW_COLD":
                    # کاوش سرد با EVoI بدون پیشین
                    scm.init_node(target.key, t_ctx)
                    prober = ActiveEVoIProber(scm)
                    for _ in range(3):
                        d, stop = prober.select_next_dose(target, t_ctx, executed_doses)
                        if stop: break
                        probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                        obs = sensors.measure_grounded_outcomes(t_items, targets=[probe], noise_sigma=0.005)
                        incurred_risk += max(0.0, -obs)
                        scm.update(target.key, t_ctx, d, obs)
                        executed_doses.append(d)
                    p_mu, s_pred, _, _ = scm.predict([target], t_ctx)

                elif reg == "GATED_1PROBE":
                    # معماری Gated: ابتدا لود پیشین به عنوان فرضیه
                    scm.init_node(target.key, t_ctx, initial_mu=src_mu, initial_cov=src_cov)

                    # گام ۱: پروب لنگر ارزان
                    probe_anchor = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - anchor_dose)
                    obs_anchor = sensors.measure_grounded_outcomes(t_items, targets=[probe_anchor], noise_sigma=0.005)
                    incurred_risk += max(0.0, -obs_anchor)
                    executed_doses.append(anchor_dose)

                    # گام ۲: ارزیابی توسط گیت تجانس علّی
                    is_consistent, reason, z_val = scm.evaluate_consistency_gate(target.key, t_ctx, anchor_dose, obs_anchor)
                    gate_decision = reason
                    z_metric = z_val

                    if is_consistent:
                        # تأیید انتقال: آپدیت ملایم پیشین با مشاهده لنگر
                        scm.update(target.key, t_ctx, anchor_dose, obs_anchor)
                    else:
                        # رد انتقال: ابطال پیشین و ریست به حالت نامطلع
                        scm.reset_to_uninformative(target.key, t_ctx)
                        scm.update(target.key, t_ctx, anchor_dose, obs_anchor)
                        # پروبینگ ترمیمی تحت EVoI برای مهار خطر
                        prober = ActiveEVoIProber(scm)
                        for _ in range(2):
                            d, stop = prober.select_next_dose(target, t_ctx, executed_doses)
                            if stop: break
                            probe_next = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                            obs_next = sensors.measure_grounded_outcomes(t_items, targets=[probe_next], noise_sigma=0.005)
                            incurred_risk += max(0.0, -obs_next)
                            scm.update(target.key, t_ctx, d, obs_next)
                            executed_doses.append(d)

                    p_mu, s_pred, _, _ = scm.predict([target], t_ctx)

                beliefs = compute_belief_distribution(p_mu, s_pred)
                verdict = resolve_governor(p_mu, s_pred, beliefs)
                u = DecoupledUtility.get_utility(verdict, gt_target)
                regret = oracle_u - u

                status_label = "✅ SAFE"
                if oracle_action == "VETO_HAZARD" and verdict == "APPROVED":
                    status_label = "🚨 HAZARD ESCAPE"
                elif oracle_action == "APPROVED" and "VETO" in verdict:
                    status_label = "⚠️ OLR LOSS"

                print(f"{target.key:<11} | {gt_target:+7.4f} | {reg:<16} | {gate_decision:<20} | {z_metric:7.2f} | {len(executed_doses):<6} | {incurred_risk:7.4f} | {verdict:<14} | {regret:7.4f} | {status_label}")

                records.append({
                    "target_context": t_ctx,
                    "target_key": target.key,
                    "gt_target": gt_target,
                    "oracle": oracle_action,
                    "regime": reg,
                    "gate_decision": gate_decision,
                    "z_metric": z_metric,
                    "probes_count": len(executed_doses),
                    "risk": incurred_risk,
                    "verdict": verdict,
                    "regret": regret,
                    "status": status_label
                })

            print(f"{'-'*185}")

    # --------------------------------------------------------------------------
    # فاز ۴: جدول کلان مقایسه ۳ سیاست
    # --------------------------------------------------------------------------
    print("\n" + "=" * 145)
    print("📊 MACRO AUDIT: GATED 1-PROBE (M18B) VS DIRECT TRANSFER (M18A) VS RAW COLD EXPLORATION")
    print("-" * 145)
    print(f"{'Target Context':<30} | {'Regime':<16} | {'Avg Probes':<11} | {'Total Risk':<12} | {'Total Regret':<13} | {'OLR (%)':<9} | {'SFR (%)'}")
    print("-" * 145)

    for t_ctx_info in target_contexts:
        t_ctx = t_ctx_info["ctx"]
        sub = [r for r in records if r["target_context"] == t_ctx]
        for reg in ["DIRECT_M18A", "RAW_COLD", "GATED_1PROBE"]:
            r_sub = [r for r in sub if r["regime"] == reg]
            avg_p = np.mean([r["probes_count"] for r in r_sub])
            t_risk = np.sum([r["risk"] for r in r_sub])
            t_reg = np.sum([r["regret"] for r in r_sub])

            n_gain = sum(1 for r in r_sub if r["oracle"] == "APPROVED")
            n_haz = sum(1 for r in r_sub if r["oracle"] == "VETO_HAZARD")

            olr = (sum(1 for r in r_sub if r["oracle"] == "APPROVED" and "VETO" in r["verdict"]) / max(1, n_gain)) * 100.0
            sfr = (sum(1 for r in r_sub if r["oracle"] == "VETO_HAZARD" and r["verdict"] == "APPROVED") / max(1, n_haz)) * 100.0

            print(f"{t_ctx:<30} | {reg:<16} | {avg_p:<11.1f} | {t_risk:<12.4f} | {t_reg:<13.4f} | {olr:<9.1f} | {sfr:<7.1f}")
        print("-" * 145)

    with open("m18b_gated_transfer_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(records), f, indent=2)
    print("\n📁 Comprehensive audit exported to 'm18b_gated_transfer_results.json'.")


if __name__ == "__main__":
    run_milestone_18b()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


/tmp/ipykernel_3022/3157657879.py:125: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 18B: GATED 1-PROBE TRANSFER & CAUSAL CONSISTENCY AUDIT
   Evaluating: Safe 1-Probe Anchor -> Causal Consistency Gate -> Dynamic Negative-Transfer Rejection

📦 PHASE 1: Harvesting Source Priors in 'train_syntactic_agreement'...
  • Source Profile [L2_head_1 ]: μ = -0.0109 | GT = -0.0169
  • Source Profile [L8_head_4 ]: μ = +0.0008 | GT = +0.0035
  • Source Profile [L22_head_0]: μ = -0.0295 | GT = -0.0129
  • Source Profile [L12_head_3]: μ = -0.0541 | GT = -0.0430
  • Source Profile [L20_mlp_0 ]: μ = -0.6545 | GT = -0.5996
  • Source Profile [L12_mlp_0 ]: μ = -0.0778 | GT = -0.0686
  • Source Profile [L16_mlp_0 ]: μ = +0.1025 | GT = +0.0650
  • Source Profile [L21_head_2]: μ = -0.0242 | GT = -0.0073

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
🎯 EVALUATION IN TARGET: Cont

In [3]:
# ==============================================================================
# MILESTONE 18C: SLOPE-CONSISTENCY GATED TRANSFER (REPAIRED AUDIT)
# ==============================================================================
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ------------------------------------------------------------------------------
# 0. HELPERS & UTILITY
# ------------------------------------------------------------------------------
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, torch.Tensor): return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None: return obj
    if hasattr(obj, "__dataclass_fields__"): return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_", "target_"]:
        if ctx_str.startswith(prefix): return ctx_str[len(prefix):]
    return ctx_str


class DecoupledUtility:
    THRESH_GAIN = 0.005
    THRESH_HAZARD = -0.05
    ALL_ACTIONS = ["APPROVED", "VETO_ZERO_GAIN", "VETO_HAZARD", "VETO_EPISTEMIC"]

    @staticmethod
    def get_utility(action: str, delta_m: float) -> float:
        if delta_m >= DecoupledUtility.THRESH_GAIN:
            if action == "APPROVED": return float(delta_m)
            if action == "VETO_EPISTEMIC": return -0.005
            if action == "VETO_ZERO_GAIN": return -0.015
            if action == "VETO_HAZARD": return -0.03
        elif delta_m < DecoupledUtility.THRESH_HAZARD:
            if action == "VETO_HAZARD": return 0.05
            if action == "VETO_EPISTEMIC": return 0.03
            if action == "VETO_ZERO_GAIN": return 0.01
            if action == "APPROVED": return float(10.0 * delta_m)
        else:
            if action == "VETO_ZERO_GAIN": return 0.005
            if action == "VETO_EPISTEMIC": return 0.002
            if action == "VETO_HAZARD": return 0.0
            if action == "APPROVED": return float(delta_m)
        return -1.0

    @staticmethod
    def get_omniscient_oracle(delta_m: float) -> Tuple[str, float]:
        utils = [DecoupledUtility.get_utility(a, delta_m) for a in DecoupledUtility.ALL_ACTIONS]
        best_idx = int(np.argmax(utils))
        return DecoupledUtility.ALL_ACTIONS[best_idx], utils[best_idx]


def compute_belief_distribution(mu: float, sig_pred: float) -> Dict[str, float]:
    sig_pred = max(sig_pred, 1e-6)
    p_haz = float(norm.cdf((DecoupledUtility.THRESH_HAZARD - mu) / sig_pred))
    p_gain = float(1.0 - norm.cdf((DecoupledUtility.THRESH_GAIN - mu) / sig_pred))
    p_zero = max(0.0, 1.0 - p_haz - p_gain)
    return {"P_gain": p_gain, "P_zero": p_zero, "P_hazard": p_haz}


def resolve_governor(mu: float, sigma_pred: float, beliefs: Dict[str, float]) -> str:
    if beliefs["P_hazard"] >= 0.50: return "VETO_HAZARD"
    if beliefs["P_gain"] >= 0.50: return "APPROVED"
    if beliefs["P_zero"] >= 0.50: return "VETO_ZERO_GAIN"
    if sigma_pred > 0.65 or max(beliefs.values()) < 0.40: return "VETO_EPISTEMIC"
    best_class = max(beliefs, key=beliefs.get)
    if best_class == "P_gain": return "APPROVED"
    if best_class == "P_hazard": return "VETO_HAZARD"
    return "VETO_ZERO_GAIN"


# ------------------------------------------------------------------------------
# 1. SENSORS & SPEC
# ------------------------------------------------------------------------------
@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(
        self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None, noise_sigma: float = 0.0
    ) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets: return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        raw_delta = float(torch.mean(int_m - clean_m).item())

        if noise_sigma > 0.0:
            return raw_delta + float(np.random.normal(0.0, noise_sigma))
        return raw_delta


# ------------------------------------------------------------------------------
# 2. REPAIRED SLOPE-CONSISTENCY BAYESIAN SELF-GRAPH
# ------------------------------------------------------------------------------
class SlopeGatedBayesianSelfGraph:
    def __init__(self, default_prior_variance: float = 0.5, base_aleatoric: float = 0.005):
        self.default_prior_variance = default_prior_variance
        self.base_aleatoric = base_aleatoric
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}

    def init_node(
        self,
        target_key: str,
        ctx_str: str,
        initial_mu: Optional[np.ndarray] = None,
        initial_cov: Optional[np.ndarray] = None
    ):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            has_imported = (initial_mu is not None and initial_cov is not None)
            if has_imported:
                mu_init = np.array(initial_mu, dtype=float)
                cov_init = np.array(initial_cov, dtype=float)
                prec_init = np.linalg.inv(cov_init)
            else:
                mu_init = np.zeros(2)
                cov_init = np.eye(2) * self.default_prior_variance
                prec_init = np.eye(2) / self.default_prior_variance

            self.component_models[target_key][family] = {
                "mu": mu_init,
                "precision": prec_init,
                "cov": cov_init,
                "aleatoric_var": self.base_aleatoric,
                "history": [],
                "imported_prior": has_imported,
                "n": 0
            }

    def reset_to_uninformative(self, target_key: str, ctx_str: str):
        family = extract_context_family(ctx_str)
        node = self.component_models[target_key][family]
        node["mu"] = np.zeros(2)
        node["cov"] = np.eye(2) * self.default_prior_variance
        node["precision"] = np.eye(2) / self.default_prior_variance
        node["imported_prior"] = False

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float):
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        node["history"].append((dose, delta_obs))

        if len(node["history"]) >= 3:
            y_vals = np.array([h[1] for h in node["history"]])
            diffs = np.diff(y_vals)
            node["aleatoric_var"] = max(self.base_aleatoric, float(0.5 * np.mean(diffs**2)))

        inv_noise = 1.0 / max(node["aleatoric_var"], 1e-6)
        prec_old = node["precision"]
        mu_old = node["mu"]

        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float, float, float]:
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        var_epi = 0.0
        var_ale = 0.0

        for t in targets:
            if t.dose == 0.0: continue
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            x_target = np.array([t.dose, t.dose**2])
            pred_delta += float(x_target @ node["mu"])
            var_epi += float(x_target @ node["cov"] @ x_target)
            var_ale += float(node["aleatoric_var"])

        total_var = var_epi + var_ale
        return pred_delta, float(np.sqrt(total_var)), float(np.sqrt(var_epi)), float(np.sqrt(var_ale))

    def evaluate_slope_consistency(
        self, target_key: str, ctx_str: str, anchor_dose: float, obs_anchor: float
    ) -> Tuple[bool, str, float]:
        """
        گیت تجانس مشتق/شیب: ارزیابی سرعت و جهت تغییرات در دوز لنگر
        """
        family = extract_context_family(ctx_str)
        node = self.component_models[target_key][family]
        if not node["imported_prior"]:
            return True, "NO_PRIOR", 0.0

        # شیب تجربی مشاهده‌شده
        observed_slope = obs_anchor / anchor_dose
        # شیب مورد انتظار تحت مدل پیشین
        expected_slope = node["mu"][0] + node["mu"][1] * anchor_dose

        # انحراف معیار شیب
        slope_vec = np.array([1.0, anchor_dose])
        slope_var = float(slope_vec @ node["cov"] @ slope_vec) + (node["aleatoric_var"] / (anchor_dose**2))
        slope_std = max(np.sqrt(slope_var), 1e-4)

        z_slope = abs(observed_slope - expected_slope) / slope_std

        # شرط ۱: تضاد آشکار در جهت شیب
        sign_conflict = (observed_slope * expected_slope < 0.0) and (abs(observed_slope) > 0.15 or abs(expected_slope) > 0.15)
        # شرط ۲: انحراف فراتر از ۱.۸ سیگما
        magnitude_conflict = (z_slope > 1.8)

        if sign_conflict or magnitude_conflict:
            return False, "REJECT_SLOPE_MISMATCH", z_slope

        return True, "ACCEPT_SLOPE_MATCH", z_slope


# ------------------------------------------------------------------------------
# 3. ACTIVE PROBER
# ------------------------------------------------------------------------------
class ActiveEVoIProber:
    def __init__(self, scm: SlopeGatedBayesianSelfGraph):
        self.scm = scm
        self.candidates = np.array([0.25, 0.40, 0.60, 0.75])
        self.cost_factor = 0.001
        self.lambda_probe_risk = 0.5
        self.eval_shifts = np.array([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
        self.eval_weights = norm.pdf(self.eval_shifts)
        self.eval_weights /= np.sum(self.eval_weights)

    def _evaluate_max_expected_utility(self, mu_star: float, sigma_star: float) -> float:
        sigma_star = max(sigma_star, 1e-6)
        samples = mu_star + sigma_star * self.eval_shifts
        action_expected_utils = []
        for a in DecoupledUtility.ALL_ACTIONS:
            eu_a = sum(w * DecoupledUtility.get_utility(a, s) for w, s in zip(self.eval_weights, samples))
            action_expected_utils.append(eu_a)
        return float(max(action_expected_utils))

    def select_next_dose(self, target: TargetSpec, ctx_str: str, executed_doses: List[float]) -> Tuple[float, bool]:
        node = self.scm.component_models[target.key][extract_context_family(ctx_str)]
        pred_mu, sig_pred, _, _ = self.scm.predict([target], ctx_str)
        current_v = self._evaluate_max_expected_utility(pred_mu, sig_pred)
        best_score = -float("inf")
        best_dose = self.candidates[0]

        pts, wts = np.polynomial.hermite.hermgauss(5)

        for d in self.candidates:
            if any(abs(d - ed) < 0.06 for ed in executed_doses): continue
            if len(executed_doses) >= 1 and pred_mu < -0.05 and d > 0.40: continue

            x_d = np.array([d, d**2])
            pred_y = float(x_d @ node["mu"])
            std_y = np.sqrt(float(x_d @ node["cov"] @ x_d) + node["aleatoric_var"])

            prec_sim = node["precision"] + (1.0 / node["aleatoric_var"]) * np.outer(x_d, x_d)
            cov_sim = np.linalg.inv(prec_sim)
            x_target = np.array([1.0, 1.0])
            sig_pred_sim = float(np.sqrt(x_target @ cov_sim @ x_target + node["aleatoric_var"]))

            expected_posterior_v = 0.0
            probe_harm_expected = 0.0
            for pt, w in zip(pts, wts):
                y_sample = pred_y + np.sqrt(2.0) * std_y * pt
                mu_sim = cov_sim @ (node["precision"] @ node["mu"] + (1.0 / node["aleatoric_var"]) * x_d * y_sample)
                mu_full_sim = float(x_target @ mu_sim)
                expected_posterior_v += (w / np.sqrt(np.pi)) * self._evaluate_max_expected_utility(mu_full_sim, sig_pred_sim)
                probe_harm_expected += (w / np.sqrt(np.pi)) * max(0.0, -y_sample)

            evoi_decision = max(0.0, expected_posterior_v - current_v)
            net_score = evoi_decision - (self.cost_factor * float(d**2)) - (self.lambda_probe_risk * probe_harm_expected)

            if net_score > best_score:
                best_score = net_score
                best_dose = float(d)

        if best_score <= 0.0 and len(executed_doses) >= 2:
            return 0.0, True
        return best_dose, False


# ------------------------------------------------------------------------------
# 4. BENCHMARK EXECUTION (MILESTONE 18C)
# ------------------------------------------------------------------------------
def run_milestone_18c():
    sensors = MechanisticSensorInterface()

    corpus = {
        "train_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ],
        "target_distractor_reasoning": [
            PromptItem("While Rome has the Colosseum, the Eiffel Tower stands in", " Paris", " Rome"),
            PromptItem("Even though Madrid is famous, the capital of Germany is", " Berlin", " Madrid"),
            PromptItem("Egypt has pyramids, but the Eiffel Tower was built in the city of", " Paris", " Cairo"),
        ],
        "target_factual_cap": [
            PromptItem("The capital of France is the city of", " Paris", " Rome"),
            PromptItem("The capital of Japan is the bustling city of", " Tokyo", " Kyoto"),
            PromptItem("The official capital city of Germany is", " Berlin", " Munich"),
            PromptItem("The historic capital of Italy is the city of", " Rome", " Milan"),
        ]
    }

    source_context = "train_syntactic_agreement"

    audit_components = [
        {"desc": "Upstream Head L2H1", "target": TargetSpec(2, "head", 1, keep_ratio=0.0)},
        {"desc": "Mid Syntactic Head L8H4", "target": TargetSpec(8, "head", 4, keep_ratio=0.0)},
        {"desc": "Late Reasoning Head L22H0", "target": TargetSpec(22, "head", 0, keep_ratio=0.0)},
        {"desc": "Critical Negative Head L12H3", "target": TargetSpec(12, "head", 3, keep_ratio=0.0)},
        {"desc": "Deep Negative MLP L20MLP", "target": TargetSpec(20, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Mid Hazard MLP L12MLP", "target": TargetSpec(12, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Positive Mid MLP L16MLP", "target": TargetSpec(16, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Late Head L21H2", "target": TargetSpec(21, "head", 2, keep_ratio=0.0)},
    ]

    target_contexts = [
        {"name": "Context_B (Distractor Reasoning)", "ctx": "target_distractor_reasoning"},
        {"name": "Context_C (Factual Capitals)", "ctx": "target_factual_cap"}
    ]

    print("\n" + "=" * 185)
    print("🔬 MILESTONE 18C: SLOPE-CONSISTENCY GATED TRANSFER & CAUSAL AUDIT")
    print("   Evaluating: Slope Derivative Gate -> Causal Negative-Transfer Elimination -> Dynamic Safe Fallback")
    print("=" * 185)

    print(f"\n📦 PHASE 1: Harvesting Source Priors in '{source_context}'...")
    source_scm = SlopeGatedBayesianSelfGraph(default_prior_variance=0.5, base_aleatoric=0.005)
    source_items = corpus[source_context]
    source_profiles = {}

    for comp in audit_components:
        target = comp["target"]
        for d in [0.15, 0.35, 0.60]:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(source_items, targets=[probe], noise_sigma=0.005)
            source_scm.update(target.key, source_context, d, obs)

        pred_mu, sig_pred, _, _ = source_scm.predict([target], source_context)
        gt_source = sensors.measure_grounded_outcomes(source_items, targets=[target], noise_sigma=0.0)
        node = source_scm.component_models[target.key][extract_context_family(source_context)]
        source_profiles[target.key] = {
            "mu": np.copy(node["mu"]),
            "cov": np.copy(node["cov"]),
            "pred_mu": pred_mu,
            "gt_source": gt_source
        }
        print(f"  • Source Profile [{target.key:<10}]: μ = {pred_mu:+7.4f} | GT = {gt_source:+7.4f}")

    records = []
    anchor_dose = 0.20  # دوز لنگر با سیگنال مناسب برای سنجش شیب

    for t_ctx_info in target_contexts:
        t_ctx_name = t_ctx_info["name"]
        t_ctx = t_ctx_info["ctx"]
        t_items = corpus[t_ctx]

        print(f"\n{'-'*185}")
        print(f"🎯 EVALUATION IN TARGET: {t_ctx_name} ({t_ctx})")
        print(f"{'-'*185}")
        print(f"{'Component':<11} | {'GT':<7} | {'Regime':<16} | {'Gate Decision':<22} | {'Z-Slope':<7} | {'Probes':<6} | {'Risk':<7} | {'Verdict':<14} | {'Regret':<7} | {'Status'}")
        print(f"{'-'*185}")

        for comp in audit_components:
            target = comp["target"]
            gt_target = sensors.measure_grounded_outcomes(t_items, targets=[target], noise_sigma=0.0)
            oracle_action, oracle_u = DecoupledUtility.get_omniscient_oracle(gt_target)
            src_mu = source_profiles[target.key]["mu"]
            src_cov = source_profiles[target.key]["cov"]

            regimes = ["DIRECT_M18A", "RAW_COLD", "SLOPE_GATED_18C"]

            for reg in regimes:
                scm = SlopeGatedBayesianSelfGraph(default_prior_variance=0.5, base_aleatoric=0.005)
                executed_doses = []
                incurred_risk = 0.0
                gate_decision = "BYPASSED"
                z_slope_val = 0.0

                if reg == "DIRECT_M18A":
                    scm.init_node(target.key, t_ctx, initial_mu=src_mu, initial_cov=src_cov)
                    p_mu, s_pred, _, _ = scm.predict([target], t_ctx)

                elif reg == "RAW_COLD":
                    scm.init_node(target.key, t_ctx)
                    prober = ActiveEVoIProber(scm)
                    for _ in range(3):
                        d, stop = prober.select_next_dose(target, t_ctx, executed_doses)
                        if stop: break
                        probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                        obs = sensors.measure_grounded_outcomes(t_items, targets=[probe], noise_sigma=0.005)
                        incurred_risk += max(0.0, -obs)
                        scm.update(target.key, t_ctx, d, obs)
                        executed_doses.append(d)
                    p_mu, s_pred, _, _ = scm.predict([target], t_ctx)

                elif reg == "SLOPE_GATED_18C":
                    scm.init_node(target.key, t_ctx, initial_mu=src_mu, initial_cov=src_cov)

                    # گام ۱: پروب لنگر شیب
                    probe_anchor = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - anchor_dose)
                    obs_anchor = sensors.measure_grounded_outcomes(t_items, targets=[probe_anchor], noise_sigma=0.005)
                    incurred_risk += max(0.0, -obs_anchor)
                    executed_doses.append(anchor_dose)

                    # گام ۲: ارزیابی تجانس مشتق/شیب
                    is_consistent, reason, z_val = scm.evaluate_slope_consistency(target.key, t_ctx, anchor_dose, obs_anchor)
                    gate_decision = reason
                    z_slope_val = z_val

                    if is_consistent:
                        # تأیید انتقال: به‌روزرسانی پیشین
                        scm.update(target.key, t_ctx, anchor_dose, obs_anchor)
                    else:
                        # ابطال پیشین و ریست به حالت نامطلع به دلیل ناسازگاری شیب
                        scm.reset_to_uninformative(target.key, t_ctx)
                        scm.update(target.key, t_ctx, anchor_dose, obs_anchor)
                        # فعال‌سازی پروبینگ ترمیمی EVoI
                        prober = ActiveEVoIProber(scm)
                        for _ in range(2):
                            d, stop = prober.select_next_dose(target, t_ctx, executed_doses)
                            if stop: break
                            probe_next = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                            obs_next = sensors.measure_grounded_outcomes(t_items, targets=[probe_next], noise_sigma=0.005)
                            incurred_risk += max(0.0, -obs_next)
                            scm.update(target.key, t_ctx, d, obs_next)
                            executed_doses.append(d)

                    p_mu, s_pred, _, _ = scm.predict([target], t_ctx)

                beliefs = compute_belief_distribution(p_mu, s_pred)
                verdict = resolve_governor(p_mu, s_pred, beliefs)
                u = DecoupledUtility.get_utility(verdict, gt_target)
                regret = oracle_u - u

                status_label = "✅ SAFE"
                if oracle_action == "VETO_HAZARD" and verdict == "APPROVED":
                    status_label = "🚨 HAZARD ESCAPE"
                elif oracle_action == "APPROVED" and "VETO" in verdict:
                    status_label = "⚠️ OLR LOSS"

                print(f"{target.key:<11} | {gt_target:+7.4f} | {reg:<16} | {gate_decision:<22} | {z_slope_val:7.2f} | {len(executed_doses):<6} | {incurred_risk:7.4f} | {verdict:<14} | {regret:7.4f} | {status_label}")

                records.append({
                    "target_context": t_ctx,
                    "target_key": target.key,
                    "gt_target": gt_target,
                    "oracle": oracle_action,
                    "regime": reg,
                    "gate_decision": gate_decision,
                    "z_slope": z_slope_val,
                    "probes_count": len(executed_doses),
                    "risk": incurred_risk,
                    "verdict": verdict,
                    "regret": regret,
                    "status": status_label
                })

            print(f"{'-'*185}")

    # --------------------------------------------------------------------------
    # فاز ۵: گزارش کلان نهایی
    # --------------------------------------------------------------------------
    print("\n" + "=" * 145)
    print("📊 MACRO AUDIT: SLOPE-GATED (M18C) VS DIRECT TRANSFER (M18A) VS RAW COLD EXPLORATION")
    print("-" * 145)
    print(f"{'Target Context':<30} | {'Regime':<16} | {'Avg Probes':<11} | {'Total Risk':<12} | {'Total Regret':<13} | {'OLR (%)':<9} | {'SFR (%)'}")
    print("-" * 145)

    for t_ctx_info in target_contexts:
        t_ctx = t_ctx_info["ctx"]
        sub = [r for r in records if r["target_context"] == t_ctx]
        for reg in ["DIRECT_M18A", "RAW_COLD", "SLOPE_GATED_18C"]:
            r_sub = [r for r in sub if r["regime"] == reg]
            avg_p = np.mean([r["probes_count"] for r in r_sub])
            t_risk = np.sum([r["risk"] for r in r_sub])
            t_reg = np.sum([r["regret"] for r in r_sub])

            n_gain = sum(1 for r in r_sub if r["oracle"] == "APPROVED")
            n_haz = sum(1 for r in r_sub if r["oracle"] == "VETO_HAZARD")

            olr = (sum(1 for r in r_sub if r["oracle"] == "APPROVED" and "VETO" in r["verdict"]) / max(1, n_gain)) * 100.0
            sfr = (sum(1 for r in r_sub if r["oracle"] == "VETO_HAZARD" and r["verdict"] == "APPROVED") / max(1, n_haz)) * 100.0

            print(f"{t_ctx:<30} | {reg:<16} | {avg_p:<11.1f} | {t_risk:<12.4f} | {t_reg:<13.4f} | {olr:<9.1f} | {sfr:<7.1f}")
        print("-" * 145)

    with open("m18c_slope_gated_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(records), f, indent=2)
    print("\n📁 Comprehensive audit exported to 'm18c_slope_gated_results.json'.")


if __name__ == "__main__":
    run_milestone_18c()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


/tmp/ipykernel_3022/3116004182.py:122: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 18C: SLOPE-CONSISTENCY GATED TRANSFER & CAUSAL AUDIT
   Evaluating: Slope Derivative Gate -> Causal Negative-Transfer Elimination -> Dynamic Safe Fallback

📦 PHASE 1: Harvesting Source Priors in 'train_syntactic_agreement'...
  • Source Profile [L2_head_1 ]: μ = -0.0109 | GT = -0.0169
  • Source Profile [L8_head_4 ]: μ = +0.0008 | GT = +0.0035
  • Source Profile [L22_head_0]: μ = -0.0295 | GT = -0.0129
  • Source Profile [L12_head_3]: μ = -0.0541 | GT = -0.0430
  • Source Profile [L20_mlp_0 ]: μ = -0.6545 | GT = -0.5996
  • Source Profile [L12_mlp_0 ]: μ = -0.0778 | GT = -0.0686
  • Source Profile [L16_mlp_0 ]: μ = +0.1025 | GT = +0.0650
  • Source Profile [L21_head_2]: μ = -0.0242 | GT = -0.0073

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
🎯 EVALUATION IN TARGET: Conte